In [2]:
# Cell 1: Setup and Load Tensors
import sys
import os
import json
import torch
from google.colab import drive

# Mount Drive and setup paths
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/Advanced Machine Learning Project'
sys.path.append(PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Load config
config_path = os.path.join(PROJECT_ROOT, 'config.json')
with open(config_path, 'r') as f:
    config = json.load(f)

# Load the saved tensors
data_folder = os.path.join(PROJECT_ROOT, 'data')
tensor_path = os.path.join(data_folder, 'processed_tensors.pt')

print("Loading tensors...")
saved_data = torch.load(tensor_path)
X_tensor = saved_data['X']
y_tensor = saved_data['y']

print(f"X shape: {X_tensor.shape} -> (Total_Windows, Sequence_Length, Num_Stocks)")
print(f"y shape: {y_tensor.shape} -> (Total_Windows)")

Loading tensors...
X shape: torch.Size([2232, 30, 19]) -> (Total_Windows, Sequence_Length, Num_Stocks)
y shape: torch.Size([2232]) -> (Total_Windows)


Chronological train val split

In [4]:
# Cell 2: Chronological Split and DataLoaders
from torch.utils.data import TensorDataset, DataLoader

# Let's use the first 80% of time for training, 20% for validation
split_idx = int(len(X_tensor) * 0.8)

X_train, y_train = X_tensor[:split_idx], y_tensor[:split_idx]
X_val, y_val = X_tensor[split_idx:], y_tensor[split_idx:]

print(f"Training windows: {len(X_train)}")
print(f"Validation windows: {len(X_val)}")

batch_size = config['training']['batch_size']

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# Create dataloaders
# CRITICAL: shuffle=False for time series! We want the model to read the market sequentially.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")

Training windows: 1785
Validation windows: 447
Train batches: 28


transformer

In [7]:
# Cell 3: Build the Model (CORRECTED)
from src.model import SP500Transformer

# Extract model hyperparams from config
d_model = config['model']['d_model']
nhead = config['model']['nhead']
num_layers = config['model']['num_layers']
dropout = config['model']['dropout']
seq_len = config['data']['sequence_length']

# FIX: Do not use the config for num_features.
# Get it dynamically from the 3rd dimension of your training tensor!
num_features = X_train.shape[2]
print(f"Dynamically detected {num_features} features (stocks) in the dataset.")

# Initialize the model
model = SP500Transformer(
    num_features=num_features,
    d_model=d_model,
    nhead=nhead,
    num_layers=num_layers,
    seq_len=seq_len,
    dropout=dropout
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Transformer Model initialized with {total_params:,} trainable parameters.")

Dynamically detected 19 features (stocks) in the dataset.
Transformer Model initialized with 1,438,337 trainable parameters.


forward pass sanity check


In [8]:
# Cell 4: The Forward Pass
# Grab a single batch of data
X_batch, y_batch = next(iter(train_loader))

print("--- Forward Pass Sanity Check ---")
print(f"1. Input batch shape: {X_batch.shape} -> (Batch_Size, Seq_Len, Features)")

# Pass data through the model
model.eval() # Set to evaluation mode so dropout layers don't interfere with our test
with torch.no_grad():
    raw_logits = model(X_batch)

print(f"2. Output shape: {raw_logits.shape} -> (Batch_Size)")
print(f"3. Raw Logits (first 5): {raw_logits[:5].numpy()}")

# The model outputs "logits" (raw, unconstrained numbers).
# We use a Sigmoid function to squash these numbers into a probability between 0% and 100%.
probabilities = torch.sigmoid(raw_logits)
print(f"4. Probabilities of an 'Up' day (first 5): {probabilities[:5].numpy()}")

print("\nSanity Check Complete! The model successfully processes cross-asset sequences.")

--- Forward Pass Sanity Check ---
1. Input batch shape: torch.Size([64, 30, 19]) -> (Batch_Size, Seq_Len, Features)
2. Output shape: torch.Size([64]) -> (Batch_Size)
3. Raw Logits (first 5): [-0.0763023  -0.07707981 -0.07431465 -0.08147137 -0.0841881 ]
4. Probabilities of an 'Up' day (first 5): [0.48093364 0.4807396  0.4814299  0.4796434  0.47896543]

Sanity Check Complete! The model successfully processes cross-asset sequences.
